> **Solutions for `2-distributions-lab.ipynb`.** Work through the lab on your own first. Open this notebook to check your answers after you have made a real attempt, or when you have been stuck on a part for a while. Many questions have more than one good answer, so compare your reasoning, not just your wording.

# Lab 2: Distributions

> **Course:** Data Science Foundations
> **Lesson:** Day 2 Lab, companion to `2-distributions.md`
> **Dataset:** `taxis` (seaborn). 6,433 New York taxi rides from March 2019, with pickup time, distance, fare, tip, and borough
> **Estimated time:** 90 minutes

## Lab Objectives

- Identify an unknown distribution from its histogram and summary statistics
- Use `scipy.stats` to draw PMFs and PDFs and check that probabilities sum (or integrate) to 1
- Simulate **Bernoulli** and **Binomial** data, the distributions behind Thursday's A/B tests
- Compute **z-scores** and test the 68-95-99.7 rule on real, skewed data
- Read a **QQ plot** to judge Normality
- Look for a Poisson process in real ride counts, and explain overdispersion

**How to work through this lab**

- Cells marked `# TODO` are yours to complete. Replace each `None` or comment with working code.
- Run every **Checkpoint** cell. If it prints `passed`, move on. If it raises an error, fix your code first.
- Each **Question** needs a short written answer in the "Your answer" cell below it.
- The dataset is different from the lesson on purpose. Adapt the lesson code; don't paste it.
- Worked solutions are in the `solutions/` folder. Attempt every part yourself before you open them: the struggle is where the learning happens.

## Part 0: Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

taxis = sns.load_dataset("taxis")
taxis["pickup"] = pd.to_datetime(taxis["pickup"])
print(f"Shape: {taxis.shape}")
taxis.head()

## Part 1: Mystery Distributions

The cell below generates four samples, **A to D**, and shuffles which letter gets which distribution. Don't try to decode the generator. Identify each sample from the data alone.

In [ ]:
_gen = np.random.default_rng(2024)
_samples = [
    _gen.normal(50, 8, 3000),
    _gen.uniform(0, 60, 3000),
    _gen.poisson(3, 3000).astype(float),
    _gen.exponential(10, 3000),
]
_order = _gen.permutation(4)
mystery = {letter: _samples[i] for letter, i in zip("ABCD", _order)}

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, (letter, x) in zip(axes.ravel(), mystery.items()):
    is_int = np.all(x == np.round(x))
    sns.histplot(x, bins=40, discrete=is_int, kde=not is_int, ax=ax, color="steelblue")
    ax.set_title(f"Sample {letter}")
plt.tight_layout()
plt.show()

In [ ]:
profile = pd.DataFrame({
    letter: {
        "mean": x.mean(),
        "var": x.var(ddof=1),
        "var_to_mean": x.var(ddof=1) / x.mean(),
        "skew": stats.skew(x),
        "min": x.min(),
        "max": x.max(),
        "all_integers": bool(np.all(x == np.round(x))),
    }
    for letter, x in mystery.items()
}).T
profile

**Question 1.** Name the distribution behind each letter and give two pieces of evidence for each. One of them is none of Uniform, Poisson, or Normal. Describe its shape and suggest what it might model.

**Model answer:**

- **D is Normal:** symmetric bell, skew ≈ 0, continuous values, mean ≈ 50.
- **A is Uniform:** flat histogram, hard edges at 0 and 60, skew ≈ 0, variance ≈ $(b-a)^2/12 = 300$.
- **B is Poisson:** whole numbers only, minimum 0, right-skewed, and **variance ≈ mean ≈ 3** (var_to_mean ≈ 1).
- **C is Exponential (none of the three):** continuous, starts at 0, strongly right-skewed (skew ≈ 2), mean ≈ standard deviation ≈ 10. It models *waiting times* between events of a Poisson process, such as the time until the next taxi pickup.

## Part 2: PMFs, PDFs, and Parameters

`scipy.stats` gives every distribution the same interface: `.pmf` (discrete) or `.pdf` (continuous), `.cdf`, `.ppf`, `.mean()`, `.var()`, `.rvs()`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

k = np.arange(0, 51)
for lam in [1, 4, 10, 30]:
    axes[0].plot(k, stats.poisson(lam).pmf(k), "o-", ms=3, label=f"λ = {lam}")
axes[0].set_title("Poisson PMF: skewed at small λ, bell-like at large λ")
axes[0].set_xlabel("k")
axes[0].legend()

x = np.linspace(-8, 8, 400)
for mu, sigma in [(0, 1), (0, 2), (3, 1)]:
    axes[1].plot(x, stats.norm(mu, sigma).pdf(x), lw=2, label=f"μ = {mu}, σ = {sigma}")
axes[1].set_title("Normal PDF: μ shifts, σ stretches")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
total_poisson = stats.poisson(4).pmf(np.arange(0, 101)).sum()
x = np.linspace(-20, 20, 20001)
total_normal = np.trapezoid(stats.norm(0, 2).pdf(x), x)
xu = np.linspace(0, 12, 120001)
total_uniform = np.trapezoid(stats.uniform(loc=2, scale=8).pdf(xu), xu)
print(f"Poisson PMF sum : {total_poisson:.6f}")
print(f"Normal PDF area : {total_normal:.6f}")
print(f"Uniform PDF area: {total_uniform:.6f}")

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
for total in (total_poisson, total_normal, total_uniform):
    assert abs(total - 1) < 1e-3
print("passed")

**Question 2.** The Uniform(2, 10) PDF has height 0.125 everywhere on [2, 10]. A PDF of a Normal with σ = 0.1 peaks at about 4. How can a density be greater than 1 when probabilities cannot be?

**Model answer:**

A density is not a probability. Only *areas* under the PDF are probabilities. A Normal with σ = 0.1 packs all its probability into a very narrow interval, so the curve must be tall (about 4) for the total area to equal 1. The Uniform(2, 10) spreads its area over a width of 8, so its height is $1/8 = 0.125$.

## Part 3: Bernoulli and Binomial (extension)

The lesson covered Uniform, Poisson, and Normal. Thursday's A/B tests use two more:

- **Bernoulli($p$):** a single yes/no trial (did this visitor sign up?). The outcome is 1 with probability $p$, else 0.
- **Binomial($n$, $p$):** the number of successes in $n$ independent Bernoulli trials. Mean $np$, variance $np(1-p)$.

In [ ]:
n_visitors, p_convert, n_days = 500, 0.08, 2000

visits = rng.binomial(1, p_convert, size=(n_days, n_visitors))   # Bernoulli trials
daily_conversions = visits.sum(axis=1)                            # Binomial counts

print(f"Observed mean {daily_conversions.mean():.2f}   theory n*p       = {n_visitors * p_convert:.2f}")
print(f"Observed var  {daily_conversions.var(ddof=1):.2f}   theory n*p*(1-p) = {n_visitors * p_convert * (1 - p_convert):.2f}")

fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(daily_conversions, discrete=True, stat="probability", color="steelblue", ax=ax)
k = np.arange(daily_conversions.min(), daily_conversions.max() + 1)
ax.plot(k, stats.binom(n_visitors, p_convert).pmf(k), "o", color="red", ms=4, label="Binomial PMF")
ax.set_title("Daily conversions: simulation vs Binomial(500, 0.08)")
ax.legend()
plt.show()

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert len(daily_conversions) == n_days
assert abs(daily_conversions.mean() - 40) < 1
print("passed")

**Question 3.** The Binomial histogram looks almost Normal. Why? How does that help when comparing two conversion rates on Thursday?

**Model answer:**

A Binomial count is a *sum* of many independent Bernoulli trials, so the Central Limit Theorem applies. With $np = 40$ and $n(1-p) = 460$ both large, the Normal approximation is excellent. It is why Thursday's two-proportion **z-test** can use the Normal distribution to compute p-values for conversion rates.

## Part 4: z-Scores and the Empirical Rule on Real Data

A **z-score** says how many standard deviations a value lies from the mean:

$$z = \frac{x - \mu}{\sigma}$$

The 68-95-99.7 rule is a statement about z-scores: for Normal data, about 68% have $|z| \le 1$, 95% have $|z| \le 2$, and 99.7% have $|z| \le 3$.

In [ ]:
def zscores(values):
    values = np.asarray(values, dtype=float)
    return (values - values.mean()) / values.std(ddof=1)


def empirical_rule_check(values):
    z = np.abs(zscores(values))
    return {k: np.mean(z <= k) for k in (1, 2, 3)}

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
z = zscores([2, 4, 6])
assert np.allclose(z, [-1, 0, 1])
print("passed")

In [ ]:
candidates = {
    "fare": taxis["fare"],
    "log(fare)": np.log(taxis["fare"]),
    "simulated Normal": rng.normal(0, 1, len(taxis)),
}
rows = {}
for name, values in candidates.items():
    shares = empirical_rule_check(values)
    rows[name] = {"within_1sd": shares[1], "within_2sd": shares[2], "within_3sd": shares[3],
                  "skew": stats.skew(values)}
rows["Normal theory"] = {"within_1sd": 0.683, "within_2sd": 0.954, "within_3sd": 0.997, "skew": 0.0}
rule_table = pd.DataFrame(rows).T.round(3)
rule_table

In [ ]:
taxis["distance_z"] = zscores(taxis["distance"])
display(taxis.loc[taxis["distance_z"].idxmax(), ["pickup", "distance", "fare", "distance_z"]])
print(f"Rides with |z| > 3: {(taxis['distance_z'].abs() > 3).sum()}")

**Question 4.** Does the empirical rule hold for `fare`? For `log(fare)`? What goes wrong if you flag "|z| > 3" as anomalies on skewed data?

**Model answer:**

No for `fare`. Because the distribution is heavily right-skewed, far more than 68% of rides sit within one standard deviation (most fares are small and similar), and the long right tail produces many more "|z| > 3" rides than the 0.3% the rule predicts. After the log transform, the shares move much closer to 68/95/99.7 and the skew drops toward 0.

On skewed data, a z > 3 rule flags too many ordinary long trips on the right while being unable to flag anything on the left (fares cannot go below 0). Transform first, or use a rule based on quantiles, such as Tukey's fences.

## Part 5: QQ Plots (extension)

The lesson warned that a bell-shaped histogram is not proof of Normality and promised a better tool. A **QQ plot** sorts your data and plots it against the values a perfect Normal would produce at the same quantiles. **Normal data falls on the straight line.** A curve bending upward on the right means a heavy right tail.

`stats.probplot(values, dist="norm", plot=ax)` draws one.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, values) in zip(axes, candidates.items()):
    stats.probplot(values, dist="norm", plot=ax)
    ax.set_title(f"QQ plot: {name}")
plt.tight_layout()
plt.show()

**Question 5.** Describe the shape of each QQ plot and what it tells you.

**Model answer:**

- **Simulated Normal:** points hug the line from end to end. This is what Normal looks like.
- **fare:** a clear upward curve. The high quantiles sit far above the line (heavy right tail), and there is a flat floor on the left from the minimum fare.
- **log(fare):** much closer to the line through the middle, with some deviation in the tails (fares cluster at fixed values such as airport flat rates). Approximately Normal, but not perfectly.

## Part 6: Looking for a Poisson Process

The number of pickups per hour sounds like a textbook Poisson variable: independent events, counted in fixed intervals. Test that idea.

In [ ]:
hours = pd.date_range(taxis["pickup"].min().floor("h"), taxis["pickup"].max().floor("h"), freq="h")
rides_per_hour = (taxis["pickup"].dt.floor("h").value_counts()
                  .reindex(hours, fill_value=0)
                  .sort_index())

print(f"Hours covered: {len(rides_per_hour)}")
print(f"Mean  : {rides_per_hour.mean():.2f}")
print(f"Var   : {rides_per_hour.var():.2f}")
print(f"Var / mean: {rides_per_hour.var() / rides_per_hour.mean():.2f}")

In [ ]:
by_hour_of_day = rides_per_hour.groupby(rides_per_hour.index.hour).mean()
fig, ax = plt.subplots(figsize=(9, 3.5))
by_hour_of_day.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Average rides per hour, by hour of day")
ax.set_xlabel("Hour of day")
plt.show()

evening = rides_per_hour[rides_per_hour.index.hour == 18]
lam = evening.mean()
print(f"18:00 slot, {len(evening)} days: mean = {lam:.2f}, var = {evening.var():.2f}, var/mean = {evening.var() / lam:.2f}")

fig, ax = plt.subplots(figsize=(9, 3.5))
sns.histplot(evening, discrete=True, stat="probability", ax=ax, color="darkorchid")
k = np.arange(0, evening.max() + 5)
ax.plot(k, stats.poisson(lam).pmf(k), "o-", color="red", ms=4, label=f"Poisson(λ = {lam:.1f})")
ax.set_title("Rides in the 18:00 hour vs a Poisson model")
ax.legend()
plt.show()

**Question 6.** Are all-day hourly counts Poisson? Why is the variance so much larger than the mean? What changed when you fixed the hour of day?

**Model answer:**

No. Across all hours, the variance is about three times the mean (26.4 vs 8.6, **overdispersion**). The rate λ is not constant: it is near zero at 4 a.m. and peaks in the evening. Mixing many different λ's in one column produces far more spread than any single Poisson.

Fixing the hour of day holds λ roughly constant, and the variance/mean ratio falls from about 3.1 to about 1.5, much closer to the Poisson signature of 1. It usually stays a little above 1 because weekday vs weekend and the weather still change the rate. When a Poisson model fails, first ask whether the rate is really constant. If it cannot be made constant, a Negative Binomial model is the usual next step.

## Part 7: Wrap-Up

In 3 to 4 sentences: which distribution would you use to model (a) the fare of a ride, (b) whether a rider tips, (c) the number of rides in the 18:00 hour, and why?

**Model answer:**

(a) Fare is positive and right-skewed. It is not Normal, but log(fare) is roughly Normal, so a log-Normal model is a reasonable start. (b) Whether a rider tips is a yes/no outcome: Bernoulli with $p$ equal to the tipping rate, and the number of tippers among $n$ rides is Binomial. (c) Rides in a fixed hour slot are counts with a roughly constant rate, so Poisson is the natural model, checked by comparing the mean and variance. Negative Binomial is the fallback if they differ a lot.

> Submit your completed notebook to the Kanban board under your name by end of day.